# Probability Distributions, Bayesian Probability, and Gradient Descent

This notebook contains the executed work for:

1. Expectation-Maximization on Galton family heights
2. Bayesian sentiment analysis on IMDb movie reviews
3. Gradient descent with SciPy-based numerical gradients


## Setup

The raw datasets are not stored in this repository. Download them from Kaggle:

- Galton heights: https://www.kaggle.com/datasets/jacopoferretti/parents-heights-vs-children-heights-galton-data
- IMDb reviews: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In Colab or Jupyter, upload either the CSV files or the Kaggle zip files before running the setup cell below.


In [ ]:
import os
import zipfile
from pathlib import Path

DATA_FILES = {
    "parents-heights-vs-children-heights-galton-data.zip": "GaltonFamilies.csv",
    "imdb-dataset-of-50k-movie-reviews.zip": "IMDB Dataset.csv",
}

for zip_name, csv_name in DATA_FILES.items():
    csv_path = Path(csv_name)
    if csv_path.exists():
        print(f"Found {csv_name}")
        continue

    if Path(zip_name).exists():
        with zipfile.ZipFile(zip_name) as zf:
            zf.extract(csv_name)
        print(f"Extracted {csv_name}")
    else:
        print(f"Missing {csv_name}")
        print(f"  Download the dataset from Kaggle, then upload {csv_name} or {zip_name}.")


In [ ]:
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import approx_fprime


## Part 1: EM Algorithm on Heights

We pool father and child heights, hide the labels, and fit a two-component Gaussian mixture using Expectation-Maximization.


In [ ]:
def load_height_data(csv_path="GaltonFamilies.csv"):
    df = pd.read_csv(csv_path)
    fathers = df.drop_duplicates("family")["father"].to_numpy()
    children = df["childHeight"].to_numpy()

    data = np.concatenate([children, fathers])
    rng = np.random.default_rng(0)
    rng.shuffle(data)
    return data, children, fathers


def gaussian_pdf(x, mu, sigma2):
    sigma2 = max(sigma2, 1e-6)
    return np.exp(-((x - mu) ** 2) / (2 * sigma2)) / np.sqrt(2 * np.pi * sigma2)


def e_step(x, mu1, mu2, sigma1_2, sigma2_2, pi1, pi2):
    p1 = pi1 * gaussian_pdf(x, mu1, sigma1_2)
    p2 = pi2 * gaussian_pdf(x, mu2, sigma2_2)
    total = np.where(p1 + p2 == 0, 1e-12, p1 + p2)
    return p1 / total, p2 / total


def m_step(x, gamma1, gamma2):
    n1, n2 = gamma1.sum(), gamma2.sum()
    mu1 = np.sum(gamma1 * x) / n1
    mu2 = np.sum(gamma2 * x) / n2
    sigma1_2 = np.sum(gamma1 * (x - mu1) ** 2) / n1
    sigma2_2 = np.sum(gamma2 * (x - mu2) ** 2) / n2
    return mu1, mu2, sigma1_2, sigma2_2, n1 / len(x), n2 / len(x)


def log_likelihood(x, mu1, mu2, sigma1_2, sigma2_2, pi1, pi2):
    mixture_density = pi1 * gaussian_pdf(x, mu1, sigma1_2) + pi2 * gaussian_pdf(x, mu2, sigma2_2)
    return np.sum(np.log(np.where(mixture_density == 0, 1e-12, mixture_density)))


In [ ]:
def run_em(x, n_iter=10):
    mu1, mu2 = np.percentile(x, [25, 75])
    sigma1_2 = sigma2_2 = np.var(x)
    pi1 = pi2 = 0.5
    history = []

    for iteration in range(n_iter + 1):
        ll = log_likelihood(x, mu1, mu2, sigma1_2, sigma2_2, pi1, pi2)
        history.append({
            "iter": iteration,
            "mu1": mu1,
            "mu2": mu2,
            "sigma1_2": sigma1_2,
            "sigma2_2": sigma2_2,
            "pi1": pi1,
            "pi2": pi2,
            "ll": ll,
        })

        if iteration < n_iter:
            gamma1, gamma2 = e_step(x, mu1, mu2, sigma1_2, sigma2_2, pi1, pi2)
            mu1, mu2, sigma1_2, sigma2_2, pi1, pi2 = m_step(x, gamma1, gamma2)

    return pd.DataFrame(history)


def classify_height(height, final_row):
    gamma_child, gamma_father = e_step(
        np.array([height]),
        final_row["mu1"], final_row["mu2"],
        final_row["sigma1_2"], final_row["sigma2_2"],
        final_row["pi1"], final_row["pi2"],
    )
    return gamma_child[0], gamma_father[0]


In [ ]:
height_data, true_children, true_fathers = load_height_data()
em_history = run_em(height_data, n_iter=10)
final_em = em_history.iloc[-1]

print(f"Pooled dataset size: {len(height_data)}")
display(em_history.round(4))

print("Final EM parameters:")
for name in ["mu1", "mu2", "sigma1_2", "sigma2_2", "pi1", "pi2"]:
    print(f"  {name}: {final_em[name]:.4f}")

p_child, p_father = classify_height(68, final_em)
print(f"\nFor a height of 68 inches: P(child)={p_child:.4f}, P(father)={p_father:.4f}")


In [ ]:
global_mean = height_data.mean()
lower_group = height_data[height_data < global_mean]
upper_group = height_data[height_data >= global_mean]

print(f"Global mean: {global_mean:.2f}")
print(f"Naive lower group mean: {lower_group.mean():.2f} (n={len(lower_group)})")
print(f"Naive upper group mean: {upper_group.mean():.2f} (n={len(upper_group)})")
print("The EM means are less biased because EM uses soft membership instead of a hard threshold.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(em_history["iter"], em_history["mu1"], marker="o", label="mu1: children")
axes[0].plot(em_history["iter"], em_history["mu2"], marker="o", label="mu2: fathers")
axes[0].set_title("EM Means over Iterations")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Height")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(em_history["iter"], em_history["ll"], marker="o", color="crimson")
axes[1].set_title("Log-Likelihood over Iterations")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Log-Likelihood")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("part1_means_and_loglikelihood.png", dpi=150)
plt.show()


## Part 2: Bayesian Sentiment Analysis

For each keyword, we estimate `P(Positive | keyword)` using counts from the IMDb review dataset.


In [ ]:
POSITIVE_KEYWORDS = ["excellent", "brilliant", "masterpiece", "love"]
NEGATIVE_KEYWORDS = ["boring", "waste", "awful", "worst"]
ALL_KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS


def load_reviews(csv_path="IMDB Dataset.csv"):
    reviews = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            reviews.append((row["review"].lower(), row["sentiment"].strip().lower()))
    return reviews


def contains_keyword(text, keyword):
    pattern = r"\b" + re.escape(keyword) + r"\b"
    return re.search(pattern, text) is not None


def bayes_for_keyword(reviews, keyword):
    total = len(reviews)
    positive_count = sum(label == "positive" for _, label in reviews)
    keyword_count = sum(contains_keyword(text, keyword) for text, _ in reviews)
    positive_keyword_count = sum(
        label == "positive" and contains_keyword(text, keyword)
        for text, label in reviews
    )

    prior = positive_count / total
    likelihood = positive_keyword_count / positive_count
    marginal = keyword_count / total
    posterior = (likelihood * prior) / marginal if marginal else 0

    return {
        "keyword": keyword,
        "P(Positive)": prior,
        "P(keyword | Positive)": likelihood,
        "P(keyword)": marginal,
        "P(Positive | keyword)": posterior,
        "keyword_count": keyword_count,
    }


In [ ]:
reviews = load_reviews()
print(f"Loaded {len(reviews)} reviews")
print(f"Positive: {sum(label == 'positive' for _, label in reviews)}")
print(f"Negative: {sum(label == 'negative' for _, label in reviews)}")

bayes_results = pd.DataFrame([bayes_for_keyword(reviews, kw) for kw in ALL_KEYWORDS])
bayes_results["keyword_type"] = ["positive"] * len(POSITIVE_KEYWORDS) + ["negative"] * len(NEGATIVE_KEYWORDS)
display(bayes_results.round(4))


In [ ]:
prior = bayes_results["P(Positive)"].iloc[0]
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["seagreen" if t == "positive" else "crimson" for t in bayes_results["keyword_type"]]

ax.bar(bayes_results["keyword"], bayes_results["P(Positive | keyword)"], color=colors)
ax.axhline(prior, color="black", linestyle="--", linewidth=1, label=f"Prior = {prior:.2f}")
ax.set_title("Bayesian Sentiment by Keyword")
ax.set_ylabel("P(Positive | keyword)")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig("part2_bayesian_sentiment_keywords.png", dpi=150)
plt.show()


## Part 3/4: Gradient Descent

We optimize the parameters of a small linear model and verify gradients numerically with SciPy.


In [ ]:
X = np.array([[1.0, 3.0], [4.0, 10.0]])
y = np.array([5.0, 6.0])

m_init = np.array([-1.0, 2.0])
b_init = np.array([1.0, 1.0])
ALPHA = 0.01
N_ITERS = 50


def predict(X, m, b):
    return X @ m + b


def mse_cost(params, X, y):
    m = params[:2]
    b = params[2:]
    return np.mean((predict(X, m, b) - y) ** 2)


def numerical_gradient(m, b, X, y, epsilon=1e-6):
    grad = approx_fprime(np.concatenate([m, b]), mse_cost, epsilon, X, y)
    return grad[:2], grad[2:]


In [ ]:
def run_gradient_descent(X, y, m, b, alpha, n_iters):
    rows = []

    for iteration in range(n_iters + 1):
        yhat = predict(X, m, b)
        error = yhat - y
        cost = np.mean(error ** 2)
        rows.append({
            "iter": iteration,
            "m1": m[0],
            "m2": m[1],
            "b1": b[0],
            "b2": b[1],
            "cost": cost,
            "prediction_1": yhat[0],
            "prediction_2": yhat[1],
        })

        if iteration < n_iters:
            grad_m, grad_b = numerical_gradient(m, b, X, y)
            m = m - alpha * grad_m
            b = b - alpha * grad_b

    return pd.DataFrame(rows)


In [ ]:
gd_history = run_gradient_descent(X, y, m_init.copy(), b_init.copy(), ALPHA, N_ITERS)

print("First four gradient descent updates:")
display(gd_history.head(5).round(6))

print("Final parameters and predictions:")
display(gd_history.tail(1).round(6))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for column in ["m1", "m2", "b1", "b2"]:
    axes[0].plot(gd_history["iter"], gd_history[column], marker="o", markersize=3, label=column)
axes[0].set_title("Parameter Values over Iterations")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Value")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(gd_history["iter"], gd_history["cost"], color="crimson", marker="o", markersize=3)
axes[1].set_title("MSE Cost over Iterations")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Cost")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("part4_gradient_descent_results.png", dpi=150)
plt.show()


## Summary

- EM separates the pooled height data into two overlapping Gaussian groups using soft assignments.
- Bayesian keyword analysis shows positive words increase `P(Positive | keyword)`, while negative words reduce it.
- Gradient descent steadily lowers the mean squared error and produces predictions closer to the target values.
